# Weak-estimates fusion — does KL fall monotonically? (GPU sweep)

The world is a fixed bivariate normal (X, Y); the solver only ever sees
**weak empirical estimates**, each computed from 5 iid draws: empirical
`E[X]`, `E[X*Y]`, `E[X^2]`, `P(X > 2)`, `P(X > Y)`, ... — dozens of them,
each carrying its honest sampling sd (oracle weighting).

**Claim under test:** `KL(fit || true)` decreases (near-)monotonically as
estimates accumulate, i.e. `expect_nll` genuinely *fuses* weak evidence.
Each seed draws one pool of 64 estimates; the m-ladder takes prefixes, so
every rung strictly adds evidence to the last. A Bayes-optimal fuser
should fall roughly like 1/m — the plot cell draws that guide.

Scoring is closed-form vs the known truth: `kl_gauss` (moment-matched
Gaussian KL) and `kl_knn` (nonparametric 1-NN KL — catches non-Gaussian
pathology). `kl_uniform_ref()` is the no-information anchor.

Composite functionals ride through **deterministic equation links**
(`xy = x * y` etc.), so `link_err` doubles as a Fermi-link load test.

All logic lives in `benchmarks/weak_estimates_experiment.py`; this
notebook is a thin shell. **Runtime -> GPU**, then Run all.

In [ ]:
# --- Setup: clone-or-pull the repo, install deps Colab lacks ---
import os, sys, subprocess

REPO = "/content/calibrated_response"
BRANCH = "main"   # must carry the weak-estimates-benchmark commit
URL = "https://github.com/amdson/calibrated_response.git"

if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, URL, REPO],
                   check=True)
else:
    subprocess.run(["git", "-C", REPO, "fetch", "--depth", "1", "origin", BRANCH],
                   check=True)
    subprocess.run(["git", "-C", REPO, "reset", "--hard", f"origin/{BRANCH}"],
                   check=True)
os.chdir(REPO)
if REPO not in sys.path:
    sys.path.insert(0, REPO)

%pip -q install optax jaxopt pydantic

import jax
jax.config.update("jax_compilation_cache_dir", f"{REPO}/.jax_cache")
jax.config.update("jax_persistent_cache_min_compile_time_secs", 1.0)
print("jax backend:", jax.default_backend(), jax.devices())
assert jax.default_backend() != "cpu", "No GPU — switch the runtime type first"

# import-guard: confirm the pulled branch carries the payload
from benchmarks.weak_estimates_experiment import (run_sweep, summarize,
                                                  full_configs, quick_configs,
                                                  expr_configs, kl_uniform_ref)
print(f"payload OK — {len(full_configs())} runs in the full grid, "
      f"uniform ref KL = {kl_uniform_ref():.3f}")

In [ ]:
# --- Sweep knobs -----------------------------------------------------------
# MODE = "quick": pool=16, m in (0, 4, 16), BOTH encodes, short fits —
#                 validates the notebook in a couple of minutes.
# MODE = "full":  aux encode — composite functionals via linked auxiliary
#                 variables (E[xy] with xy = x*y).  pool=64 per seed,
#                 m-ladder (0, 2, 4, 8, 16, 32, 64), 3 seeds x 3000 steps.
# MODE = "expr":  the A/B arm — IDENTICAL pools, but composite functionals
#                 constrain the 2-D joint directly as expressions
#                 (E[x * y], P(x - y > 0)): no aux variables, no links to
#                 leak.  If the aux arm's corr_err plateau is the link
#                 leak, this arm fixes it.
MODE = "full"

CONFIGS = (quick_configs() if MODE == "quick"
           else expr_configs() if MODE == "expr"
           else full_configs())
OUT = "results/weak_estimates.jsonl"
print(f"{MODE}: {len(CONFIGS)} runs -> {OUT}")

In [ ]:
# --- Run the sweep (resumable: rows append to the OUT jsonl) ----------------
# fits: _key(row) -> {builder, pool, used, samples, pts} for every run
# executed THIS call (resumed rows have no live sampler).
rows, fits = run_sweep(CONFIGS, out_path=OUT)

In [ ]:
# --- Summarize ---------------------------------------------------------------
# Want: kl columns falling with m, and the per-seed ladder-step
# monotonicity fractions near 100%.
summarize(OUT)

In [ ]:
# --- KL vs m: the headline plot ----------------------------------------------
import json as _json
import numpy as np
import matplotlib.pyplot as plt
from benchmarks.weak_estimates_experiment import kl_uniform_ref

with open(OUT) as fh:
    R = [_json.loads(l) for l in fh]
# keep only rows matching this MODE's grid (all modes share the jsonl)
want = (CONFIGS[0]["pool_size"], CONFIGS[0].get("steps", 3000),
        CONFIGS[0].get("encode", "aux"))
R = [r for r in R if (r["pool_size"], r["steps"],
                      r.get("encode", "aux")) == want]

fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, metric in zip(axes, ("kl_gauss", "kl_knn")):
    by_seed = {}
    for r in R:
        by_seed.setdefault(r["seed"], []).append(r)
    for s, g in sorted(by_seed.items()):
        g = sorted(g, key=lambda r: r["m"])
        # m=0 has no log-x home; plot it at m=1 with an open marker
        ms = [max(r["m"], 1) for r in g]
        ax.plot(ms, [r[metric] for r in g], "o-", ms=4, alpha=0.8,
                label=f"seed {s}")
        if g and g[0]["m"] == 0:
            ax.plot(ms[0], g[0][metric], "o", mfc="none", ms=9, c="gray")
    mm = np.array(sorted({max(r["m"], 1) for r in R}))
    anchor = np.median([r[metric] for r in R if r["m"] <= 2] or [1.0])
    ax.plot(mm, anchor * mm[0] / mm, "k--", lw=1, alpha=0.5, label="~1/m")
    ax.axhline(kl_uniform_ref(), c="gray", ls=":", lw=1, label="uniform box")
    ax.set_xscale("log"); ax.set_yscale("log")
    ax.set_xlabel("estimates fused (m)"); ax.set_title(metric)
axes[0].set_ylabel("KL(fit || true)  [nats]")
axes[0].legend(fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
# --- Inspect one fitted sampler ------------------------------------------------
# Contours of the TRUE density under the fitted (x, y) samples, plus the
# worst-fitted constraints — where does residual KL live?
import numpy as np
import matplotlib.pyplot as plt
from benchmarks.weak_estimates_experiment import MU, COV, _true_logpdf

key = sorted(fits)[-1]          # (pool_size, m, steps, K, seed)
fit = fits[key]
print("inspecting", key)
b, pts = fit["builder"], fit["pts"]

gx, gy = np.meshgrid(np.linspace(-3.5, 4.5, 120), np.linspace(-6, 5, 120))
gz = np.exp(_true_logpdf(np.stack([gx.ravel(), gy.ravel()], 1))).reshape(gx.shape)
plt.figure(figsize=(6, 5))
plt.hist2d(pts[:, 0], pts[:, 1], bins=80, cmap="viridis")
plt.contour(gx, gy, gz, levels=6, colors="w", linewidths=0.8, alpha=0.8)
plt.plot(*MU, "r+", ms=14, mew=2)
plt.xlabel("x"); plt.ylabel("y")
plt.title(f"fit samples vs true contours — {key}")
plt.tight_layout(); plt.show()

rep = sorted(b.constraint_report(), key=lambda c: -abs(c["error_rel"]))
for c in rep[:12]:
    print(f"{c['id']:>20}  target={c['target']:.3f} "
          f"fitted={c['fitted']:.3f}  err_rel={c['error_rel']:+.3f}")

In [ ]:
# --- Pairwise (corner) plot incl. the aux variables ----------------------------
# Sanity on the deterministic links: xy should hug x*y, xx should be the
# folded parabola of x.  Dashed lines mark the TRUE means.
from calibrated_response.maxent_sampler import plot_pairwise
from benchmarks.weak_estimates_experiment import ORACLE

fit = fits[key]
b = fit["builder"]
# expr-encoded fits have no aux variables — show what exists
NAMES = [n for n in ("x", "y", "xy", "xx") if n in b.var_name_to_idx]
sites = [b.var_name_to_idx[n] for n in NAMES]
plot_pairwise(b.model, b.params, sites=sites, names=NAMES,
              n_samples=30_000, seed=11, bins=50,
              threshold={n: ORACLE[n][0] for n in NAMES});

In [ ]:
# --- Download results (unzip into results/ locally to merge) ---
import shutil
shutil.make_archive("/content/weak_estimates_results", "zip", "results")
try:
    from google.colab import files
    files.download("/content/weak_estimates_results.zip")
except ImportError:
    print("not on Colab — results in results/")